# ETH Transaction Graph — Global & Daily Top Node Ranking

**Part of:** Ethereum Topological Anomaly Detection (ETH-TAD)  
**Paper:** Ofori-Boateng et al. (2021) - arXiv:2106.01806

## Description

Processes ETH and ERC20 transaction data (weekly parquet chunks), applies user-defined
filters, builds a weighted graph, and offers two ranking modes:

**Mode A — Edge-based ranking**  
Selects the top-N edges by weight and collects the nodes connected to those edges.

**Mode B — Centrality-based ranking**  
Ranks nodes directly by a graph centrality metric (default: PageRank).
Outputs the top-N central nodes per filter per period.

## Prerequisites
- Completed notebooks: 1a, 1b (data download)
- Required data: ETH and ERC20 transaction parquet files

## Outputs
- `ranking/global_top_nodes.parquet` — global rankings (both modes, all filters)
- `ranking/daily/daily_ranking_YYYY-MM_WN.parquet` — edge-based daily rankings
- `ranking/daily/daily_centrality_YYYY-MM_WN.parquet` — centrality daily rankings

## Setup

In [5]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

import pandas as pd
from pathlib import Path

from ranking_functions import (
    check_directories,
    run_global_ranking,
    run_daily_ranking,
    run_global_centrality_ranking,
    run_daily_centrality_ranking,
)

print('Imports OK ✓')

Imports OK ✓


## Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════

YEAR = 2020

ETH_DIR      = Path(f'data/{YEAR}/eth_tx_value_output/weekly')
ERC20_DIR    = Path(f'data/{YEAR}/erc20_tx_value_output/weekly')
RANKING_DIR  = Path(f'data/ranking/{YEAR}')
GLOBAL_FILE  = RANKING_DIR / 'global_top_nodes.parquet'
DAILY_DIR    = RANKING_DIR / 'daily'
INDEX_FILE   = RANKING_DIR / 'source_index.json'

RANKING_DIR.mkdir(parents=True, exist_ok=True)
DAILY_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════
# DATE RANGE
# ══════════════════════════════════════════════════════════════════════════

START_DATE = pd.Timestamp(f'{YEAR}-01-01')
END_DATE   = pd.Timestamp(f'{YEAR}-12-31')

# ══════════════════════════════════════════════════════════════════════════
# RANKING PARAMETERS
# ══════════════════════════════════════════════════════════════════════════

TOP_N = 2000  # How many top edges/nodes to consider per day/period

# ══════════════════════════════════════════════════════════════════════════
# DEAD ADDRESSES (to filter out)
# ══════════════════════════════════════════════════════════════════════════

DEAD_ADDRESSES = [
    None,
    "\\N",
]

# ══════════════════════════════════════════════════════════════════════════
# EDGE-BASED RANKING
# ══════════════════════════════════════════════════════════════════════════

# total_gas_fees = gas_used * effective price paid per unit gas (ETH burned
# on gas), added alongside tx_count/tx_value. Only populated for ETH-native
# rows (eth_data_fetcher.py) -- ERC20 transfer events (erc20_data_fetcher.py)
# don't carry gas info in Xatu's schema (gas belongs to the parent tx, not
# the individual decoded Transfer event), so any filter that mixes ETH and
# ERC20 activity (most of the "_ALL" filters below) will UNDER-count
# total_gas_fees for the ERC20-sourced share of an edge -- NaN is skipped in
# the groupby sum, not treated as zero, but the result is still a partial
# sum for those edges. This is exact/complete for the "_ETH_only" filters
# (erc20=='ETH' excludes ERC20 rows entirely) -- rank by total_gas_fees on
# those, not on the "_ALL" filters, until/unless ERC20 gas attribution
# (join canonical_execution_erc20_transfers -> canonical_execution_transaction
# on transaction_hash, then decide how to split one tx's fee across
# multiple transfer events in it) is worked out.
METRICS   = ['tx_count', 'tx_value', 'total_gas_fees']
KEEP_COLS = ['date', 'from_addr', 'to_addr', 'tx_count', 'tx_value', 'total_gas_fees']

# ══════════════════════════════════════════════════════════════════════════
# CENTRALITY-BASED RANKING
# ══════════════════════════════════════════════════════════════════════════

# NOTE: the built-in run_daily_centrality_ranking/centrality_rank_nodes in
# ranking_functions.py builds the FULL unbounded daily graph (hundreds of
# thousands of nodes for busy filters) before pruning by subgraph_top_n --
# the prune itself doesn't avoid that cost, and the graph gets rebuilt fresh
# per centrality metric. Empirically this is the likely cause of centrality
# runs that took a week: on a real busy day/filter this graph is 500k+
# nodes, ~7-8s just to build+prune ONCE, times every (filter, day, metric)
# combination. The actual multi-year centrality run uses a separate,
# efficient path (build the graph once per (filter, day, weight_col) from
# the SAME top-N-edge selection as Mode A, then compute every requested
# metric on that one graph) -- see run_ranking_all_years.py, not the cells
# below, which are left as the original single-year reference/manual-run
# path.
CENTRALITY_METRICS = ['page_rank', 'strength', 'k_core', 'clustering']  # eigenvector excluded: crashes on these graphs (naturally disconnected after top-N edge selection -- 100s of components typical); betweenness_approx excluded: real cost even with the efficient path (~3-5hrs for the whole dataset at one weight column), left out for this round
CENTRALITY_WEIGHT_COLS = ['tx_count', 'tx_value', 'total_gas_fees']
CENTRALITY_WEIGHT_COL = 'tx_count'  # single-weight default for the manual cells below
SUBGRAPH_TOP_N = None  # not used by the efficient path -- top-N-edge selection already bounds graph size

print(f'Year: {YEAR}')
print(f'Date range: {START_DATE.date()} → {END_DATE.date()}')
print(f'Top N: {TOP_N}')
print(f'Output: {RANKING_DIR}')
print('\nConfiguration set ✓')

## Filter Definitions

Define transaction filters. Each filter receives a DataFrame with an 'erc20' column  
(ETH-native rows have `erc20='ETH'`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BASIC FILTERS
# ══════════════════════════════════════════════════════════════════════════

FILTERS = {
    'contract_txs_ETH_only': lambda d: (d['tx_value'] == 0) & (d['erc20']=='ETH'),
    'simple_txs_ETH_only':   lambda d: (d['tx_value'] >  0) & (d['erc20']=='ETH'),
    'contract_factory_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['erc20']=='ETH'),
    'contract_nonFactory_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']<100) & (d['erc20']=='ETH'),
    'contract_highInput_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=500) & (d['erc20']=='ETH'),
    'contract_mediumInput_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['total_input_bytes']<500) & (d['erc20']=='ETH'),
    'contract_txs_ALL': lambda d: (d['tx_value'] == 0),
    'simple_txs_ALL':   lambda d: (d['tx_value'] >  0),
    'contract_factory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100),
    'contract_nonFactory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']<100),
    'contract_highInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=500),
    'contract_mediumInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['total_input_bytes']<500),
    # ERC20_only: the complement of ETH_only -- isolates pure ERC20 activity
    # (excludes native ETH rows entirely), as opposed to _ALL (ETH+ERC20
    # mixed). Added to directly test whether ERC20 contract-call activity
    # is independently noisy or just dilutes a clean ETH-only signal when
    # mixed in -- see the contract_txs_ALL vs. contract_txs_ETH_only
    # finding this is meant to help explain.
    'contract_txs_ERC20_only': lambda d: (d['tx_value'] == 0) & (d['erc20']!='ETH'),
    'simple_txs_ERC20_only':   lambda d: (d['tx_value'] >  0) & (d['erc20']!='ETH'),
}


# ══════════════════════════════════════════════════════════════════════════
# OPTIONAL: ERC20 TOKEN FILTERS
# ══════════════════════════════════════════════════════════════════════════

# Uncomment to add specific ERC20 token filters

# erc20_addresses = {
#     "USDT": "0xdac17f958d2ee523a2206206994597c13d831ec7",
#     "USDC": "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48",
#     "LINK": "0x514910771af9ca656af840dff83e8264ecf986ca",
#     "UNI":  "0x1f9840a85d5af5bf1d1762f925bdaddc4201f984",
#     "WBTC": "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599",
#     "DAI":  "0x6b175474e89094c44da98b954eedeac495271d0f",
# }

# addresses_lower = {k: v.lower() for k, v in erc20_addresses.items()}

# erc20filters = {
#     symbol: (lambda addr: (lambda d: d["erc20"].str.lower() == addr))(address)
#     for symbol, address in addresses_lower.items()
# }

# # Merge with basic filters
# FILTERS = {**FILTERS, **erc20filters}

print(f'Filters defined: {list(FILTERS.keys())}')

## Sanity Check Data Directories

In [8]:
assert check_directories(ETH_DIR, ERC20_DIR), 'Fix the folder paths above before continuing.'

[OK]    ETH: 53 parquet files in /Users/uri/Desktop/dev/nuru/ethTDA/ETH Anomaly Detection/data/2020/eth_tx_value_output/weekly
[OK]    ERC20: 53 parquet files in /Users/uri/Desktop/dev/nuru/ethTDA/ETH Anomaly Detection/data/2020/erc20_tx_value_output/weekly


## Run Rankings

Uncomment whichever runs you want. Order shown is recommended (cheapest first).

### Mode A: Edge-Based Ranking

In [9]:
# Global edge-based ranking
# run_global_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     metrics=METRICS,
#     keep_cols=KEEP_COLS,
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
# )

In [ ]:
# Daily edge-based ranking
run_daily_ranking(
    filters=FILTERS,
    eth_dir=ETH_DIR,
    erc20_dir=ERC20_DIR,
    daily_dir=DAILY_DIR,
    index_file=INDEX_FILE,
    metrics=METRICS,
    keep_cols=KEEP_COLS,
    start=START_DATE,
    end=END_DATE,
    top_n=TOP_N,
    dead_ads=DEAD_ADDRESSES,
)

[index] Loaded cache from data/ranking/2020/source_index.json  (60 weeks, built for 2020-01-01 - 2020-12-31)
[daily] 60 output weeks to process.


  Reading sources:   0%|                               | 0/4 [00:00<?, ?file/s]

### Mode B: Centrality-Based Ranking

**Available centrality metrics:**
- `page_rank` — weighted PageRank (fast, default choice)
- `degree` — unweighted degree
- `strength` — weighted degree
- `k_core` — k-core number
- `hits_hub` — hub scores
- `hits_authority` — authority scores
- `eigenvector` — eigenvector centrality
- `clustering` — weighted clustering coefficient
- `betweenness_approx` — betweenness centrality (slow, use `subgraph_top_n`)

**Edge mode:** Append `_edge` to any metric (e.g., `page_rank_edge`) to rank edges instead of nodes.

In [ ]:
# Global centrality ranking — fast metrics (no subgraph needed)
# run_global_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     centrality_metrics=['page_rank', 'k_core', 'strength'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=SUBGRAPH_TOP_N,
# )

In [ ]:
# Global centrality ranking — betweenness (MUST use subgraph_top_n)
# run_global_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     centrality_metrics=['betweenness_approx'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=50_000,  # Required for betweenness
# )

In [ ]:
# Daily centrality ranking — fast metrics
# run_daily_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     daily_dir=DAILY_DIR,
#     index_file=INDEX_FILE,
#     centrality_metrics=['page_rank', 'k_core', 'strength'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=SUBGRAPH_TOP_N,
# )

In [ ]:
# Daily centrality ranking — clustering coefficient
run_daily_centrality_ranking(
    filters=FILTERS,
    eth_dir=ETH_DIR,
    erc20_dir=ERC20_DIR,
    daily_dir=DAILY_DIR,
    index_file=INDEX_FILE,
    centrality_metrics=['clustering'],
    start=START_DATE,
    end=END_DATE,
    top_n=TOP_N,
    dead_ads=DEAD_ADDRESSES,
    weight_col=CENTRALITY_WEIGHT_COL,
    subgraph_top_n=10_000,
)

## Inspection

View the generated rankings.

In [6]:
# Global ranking
if GLOBAL_FILE.exists():
    g = pd.read_parquet(GLOBAL_FILE)
    print(f'Global ranking shape: {g.shape}')
    display(g.head(10))
    display(
        g.groupby(['filter', 'ranking_metric', 'start_date', 'end_date'])
         .size().rename('node_count').reset_index()
    )
else:
    print('No global ranking file found.')

No global ranking file found.


In [7]:
# Daily ranking
daily_files = sorted(DAILY_DIR.glob('*.parquet'))
print(f'Daily ranking files: {len(daily_files)}')

if daily_files:
    sample = pd.read_parquet(daily_files[0])
    print(f'\nSample file: {daily_files[0].name}  ({len(sample):,} rows)')
    display(sample.head(10))

    frames   = [pd.read_parquet(f)[['filter', 'ranking_metric', 'date']] for f in daily_files]
    coverage = pd.concat(frames, ignore_index=True)
    coverage['date'] = pd.to_datetime(coverage['date'])
    print('\nDate coverage per (filter, metric):')
    display(
        coverage.groupby(['filter', 'ranking_metric'])
                .agg(dates=('date', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max'))
                .reset_index()
    )

Daily ranking files: 60

Sample file: daily_ranking_2020-01_W1.parquet  (145,496 rows)


,address,filter,ranking_metric,top_rank,bottom_rank,date
0,0x0000000000000000000000000000000000000000,contract_txs_ETH_only,tx_count,714,714,2020-01-01
1,0x000000000000541e251335090ac5b47176af4f7e,contract_txs_ETH_only,tx_count,628,628,2020-01-01
2,0x0000000000b3f879cb30fe243b4dfee438691c04,contract_txs_ETH_only,tx_count,310,678,2020-01-01
3,0x0000000000c90bc353314b6911180ed7e06019a9,contract_txs_ETH_only,tx_count,415,766,2020-01-01
4,0x00000000c0293c8ca34dac9bcc0f953532d34e4d,contract_txs_ETH_only,tx_count,15,15,2020-01-01
5,0x0000004e4f4d35154f12386b64ca2449b8724607,contract_txs_ETH_only,tx_count,925,925,2020-01-01
6,0x000007222caeb29694719e804b24fb3ee5116a8a,contract_txs_ETH_only,tx_count,547,547,2020-01-01
7,0x000042420e7913cefee6988cf825c4a2b739684d,contract_txs_ETH_only,tx_count,224,224,2020-01-01
8,0x000042427b681ccf127a5112923322629183b109,contract_txs_ETH_only,tx_count,215,215,2020-01-01
9,0x00004242f7b5c47ae90eaf0f1c2dddf630774c80,contract_txs_ETH_only,tx_count,212,212,2020-01-01



Date coverage per (filter, metric):


,filter,ranking_metric,dates,min_date,max_date
0,contract_factory_ALL,tx_count,366,2020-01-01,2020-12-31
1,contract_factory_ALL,tx_value,366,2020-01-01,2020-12-31
2,contract_highInput_ALL,tx_count,366,2020-01-01,2020-12-31
3,contract_highInput_ALL,tx_value,366,2020-01-01,2020-12-31
4,contract_mediumInput_ALL,tx_count,366,2020-01-01,2020-12-31
5,contract_mediumInput_ALL,tx_value,366,2020-01-01,2020-12-31
6,contract_nonFactory_ALL,tx_count,366,2020-01-01,2020-12-31
7,contract_nonFactory_ALL,tx_value,366,2020-01-01,2020-12-31
8,contract_txs_ALL,tx_count,366,2020-01-01,2020-12-31
9,contract_txs_ALL,tx_value,366,2020-01-01,2020-12-31
